In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
# from scipy import stats
import pickle
import json
import copy

In [ ]:
# for plotting

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
rundirs = [
    'goamazon_2pulse.largedom.r20251008.rerun',
    'goamazon_2pulse.largedom.ehe1.r20251030.rerun',
]
casenames = [
    'CTL',
    'EHEall',
]
casecolors = [ 
    'black',
    'green',
]
stats = [] 
for rundir in rundirs:
    with open(f'{rundir}/pkl/csd_stats.pkl', 'rb') as f:
        stats.append(pickle.load(f))

In [ ]:
minmf = 0
maxmf = np.max([np.max(s[0]) for s in stats]) + 10.0
maxmf = np.log10(maxmf)
print(minmf, maxmf)

In [ ]:
nx, ny, nz, nt = 512, 512, 100, 241
dts = 0.5 # minute
dx = 250 # m
dy = 250 # m
dz = 50 # m
grid_vol = dx*1.0e-3*dy*1.0e-3*dz*1.0e-3 # km**3
z = np.arange(dz/2, 5000., dz)
t = np.arange(0, 241)*dts # minutes
ti = np.arange(-0.5, 241., 1.)*dts
zi = np.arange(0., 5001., dz)

In [ ]:
def binned_wq(csd_stats, rundir, bins):
    global nx, ny, nz, nt
    print(nx, ny, nz, nt)
    with open(f'{rundir}/pkl/plume_all_wq.pkl', 'rb') as f:
        wq = pickle.load(f)
    clipped_mf = csd_stats[0] 
    attached_ind = csd_stats[-1]
    bin_ids = np.digitize(np.log10(clipped_mf), bins)
    nbins = len(bins) - 1
    factor = 1.0/float(nx*ny*nt)
    sum_wq = np.zeros((nbins, nz))
    mean_wq = np.zeros((nbins, nz))
    mean_wq2 = np.zeros((nbins, nz))
    total_count = 0
    for bnm in range(nbins):
        bn = bnm + 1
        cn = attached_ind[bin_ids==bn]
        total_count += len(cn)
        print(bn, len(cn))
        if len(cn) > 0:
            sum_wq[bnm, :] = (wq[cn, :, :].sum(axis=1)*factor).sum(axis=0)
            mean_wq[bnm, :] = (wq[cn, :, :].sum(axis=1)*factor).mean(axis=0)
            # mean_wq2[bnm, :] = (wq[cn, :, :].sum(axis=1)*factor).sum(axis=0)/len(cn)
    # return sum_wq, mean_wq, mean_wq2 
    return sum_wq, mean_wq

In [ ]:
def binned_qc(csd_stats, casename, bins):
    global nx, ny, nz, nt
    print(nx, ny, nz, nt)
    with open(f'{casename}/pkl/plume_all_qc.pkl', 'rb') as f:
        qc = pickle.load(f)
    clipped_mf = csd_stats[0] 
    attached_ind = csd_stats[-1]
    bin_ids = np.digitize(np.log10(clipped_mf), bins)
    nbins = len(bins) - 1
    factor = 1.0/float(nx*ny*nt)
    # factor = 1.0/float(nt)
    sum_qc = np.zeros((nbins, nz))
    mean_qc = np.zeros((nbins, nz))
    total_count = 0
    for bnm in range(nbins):
        bn = bnm + 1
        cn = attached_ind[bin_ids==bn]
        total_count += len(cn)
        print(bn, len(cn))
        if len(cn) > 0:
            sum_qc[bnm, :] = (qc[cn, :, :].sum(axis=1)*factor).sum(axis=0)
            mean_qc[bnm, :] = (qc[cn, :, :].sum(axis=1)*factor).mean(axis=0)
    return sum_qc, mean_qc 

In [ ]:
nbins = 15 
bins = np.linspace(minmf, maxmf, nbins+1)
sum_wqs = []
mean_wqs = []
sum_qcs = []
mean_qcs = []
for s, d in zip(stats, rundirs):
    sum_wq, mean_wq = binned_wq(s, d, bins)
    sum_wqs.append(sum_wq)
    mean_wqs.append(mean_wq)
    sum_qc, mean_qc = binned_qc(s, d, bins)
    sum_qcs.append(sum_qc)
    mean_qcs.append(mean_qc)

In [ ]:
def compare_wq(sum_wqs, mean_wqs, casenames, casecolors, minmf, maxmf, plot_top=4.0, nbins=9):

    global zi
    bins = np.linspace(minmf, maxmf, nbins+1)

    # Create comparison plots for all EHE experiments
    fig, ax = plt.subplots(1, 1, figsize=(8, 12))
    
    sum_wq_diff = (sum_wqs[1] - sum_wqs[0])*1.0e3  # g/kg/m^2/s
    print(f"{casenames[1]}: max={sum_wq_diff.max()}, min={sum_wq_diff.min()}")
    
    levels = np.concatenate([np.arange(-14, -0.1, 2), [-0.1], [-0.01, 0.01], [0.1], np.arange(2, 14.1, 2)])
    cmap = copy.deepcopy(mpl.cm.bwr)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm = ax.pcolormesh(bins, zi*1.0e-3, sum_wq_diff.T, norm=norm, cmap=cmap, shading='flat')
    # ax.axhline(0.6125, color='black', linestyle='--', linewidth=1.0)
    ax.set_ylim(0, plot_top)
    ax.set_xlabel(r'$\log_{10}\left<M_b\right>$ (kg/s)', fontsize=18)
    ax.set_title(f"{casenames[1]} - CTL", fontsize=18)
        
    ax.set_ylabel('Height (km)', fontsize=18)
    # Adjust spacing to prevent colorbar overlap
    fig.subplots_adjust(right=0.85)
    cbar_ax = fig.add_axes([0.87, ax.get_position().y0, 0.02, ax.get_position().height])
    cbar = plt.colorbar(cm, cax=cbar_ax)
    cbar.set_ticks(levels)
    cbar.ax.tick_params(labelsize=16)
    cbar.set_label(r"$\overline{w'q'}$ diff (10$^{-3}$ g kg$^{-1}$ m s$^{-1}$)", fontsize=18)
    
    # plt.tight_layout()
    plt.show()

    bins_to_plot = [10, 11, 12, 13 ,14]
    fig, axs = plt.subplots(1, len(bins_to_plot), figsize=(18, 8))
    axs = axs.flatten()

    # Adjust spacing to make panels closer together
    fig.subplots_adjust(left=0.08, right=0.98, wspace=0.05)
    for iax, binno in enumerate(bins_to_plot):
        ax = axs[iax]
        ax.plot(mean_wqs[0][binno,:]*1e3, z*1.0e-3, color=casecolors[0], linewidth=2.5, label=casenames[0])
        for idx, m in enumerate(mean_wqs[1:]):
            ax.plot(m[binno,:]*1e3, z*1.0e-3, color=casecolors[idx+1], linewidth=2.5, label=casenames[idx+1])
        ax.set_ylim((0, plot_top))
        ax.set_xlim()
        ax.grid(alpha=0.3)
        if iax != 0:
            ax.set_yticklabels([])
        
        # Add labels only to left column and bottom row
        if iax == 0:
            ax.set_ylabel('Height (km)')

        # Simpler title with bin range
        ax.set_title(f'Bin {binno+1}\n'+fr'{10**(bins[binno]):.0f}$\minus${10**(bins[binno+1]):.0f} kg/s', fontsize=14)
        
        # Legend only in first subplot
        if iax == 0:
            ax.legend(loc='upper right', fontsize=11, frameon=True, framealpha=0.9)

    fig.suptitle(r"Mean $\overline{w'q'}$ (10$^{-3}$ g kg$^{-1}$ m s$^{-1}$)", fontsize=24)
    plt.tight_layout()
    plt.show()

    bins_to_plot = [10, 11, 12, 13 ,14]
    fig, axs = plt.subplots(1, len(bins_to_plot), figsize=(18, 8))
    axs = axs.flatten()

    # Adjust spacing to make panels closer together
    fig.subplots_adjust(left=0.08, right=0.98, wspace=0.05)
    for iax, binno in enumerate(bins_to_plot):
        ax = axs[iax]
        ax.plot(sum_wqs[0][binno,:]*1e3, z*1.0e-3, color=casecolors[0], linewidth=2.5, label=casenames[0])
        for idx, m in enumerate(sum_wqs[1:]):
            ax.plot(m[binno,:]*1e3, z*1.0e-3, color=casecolors[idx+1], linewidth=2.5, label=casenames[idx+1])
        ax.set_ylim((0, plot_top))
        ax.set_xlim()
        ax.grid(alpha=0.3)
        if iax != 0:
            ax.set_yticklabels([])
        
        # Add labels only to left column and bottom row
        if iax == 0:
            ax.set_ylabel('Height (km)')

        # Simpler title with bin range
        ax.set_title(f'Bin {binno+1}\n'+fr'{10**(bins[binno]):.0f}$\minus${10**(bins[binno+1]):.0f} kg/s', fontsize=14)
        
        # Legend only in first subplot
        if iax == 0:
            ax.legend(loc='upper right', fontsize=11, frameon=True, framealpha=0.9)

    fig.suptitle(r"Sum $\overline{w'q'}$ (10$^{-3}$ g kg$^{-1}$ m s$^{-1}$)", fontsize=24)
    plt.tight_layout()
    plt.show()

    return

In [ ]:
def compare_qc(sum_qcs, mean_qcs, casenames, casecolors, minmf, maxmf, plot_top=4.0, nbins=9):

    global zi
    bins = np.linspace(minmf, maxmf, nbins+1)

    # Create comparison plots for all EHE experiments
    fig, ax = plt.subplots(1, 1, figsize=(8, 12))
    
    sum_qc_diff = (sum_qcs[1] - sum_qcs[0])*1.0e3  # g/kg/m^2/s
    print(f"{casenames[1]}: max={sum_qc_diff.max()}, min={sum_qc_diff.min()}")
    
    levels = np.concatenate([np.arange(-2.5, -0.01, 0.5), [-0.01, 0.01], np.arange(0.5, 2.51, 0.5)])

    # print(levels)
    cmap = copy.deepcopy(mpl.cm.bwr)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm = ax.pcolormesh(bins, zi*1.0e-3, sum_qc_diff.T, norm=norm, cmap=cmap, shading='flat')
    # ax.axhline(0.6125, color='black', linestyle='--', linewidth=1.0)
    ax.set_ylim(0, plot_top)
    ax.set_xlabel(r'$q_c$ (10$^{-3}$ g/kg)', fontsize=18)
    ax.set_title(f"{casenames[1]} - CTL", fontsize=18)
    
    ax.set_ylabel('Height (km)', fontsize=18)

    fig.subplots_adjust(right=0.85)
    cbar_ax = fig.add_axes([0.87, ax.get_position().y0, 0.02, ax.get_position().height])
    cbar = plt.colorbar(cm, cax=cbar_ax)
    cbar.set_ticks(levels)
    cbar.ax.tick_params(labelsize=16)
    cbar.set_label(r"$q_c$ diff (10$^{-3}$ g kg$^{-1}$)", fontsize=18)
    
    # plt.tight_layout()
    plt.show()

    bins_to_plot = [10, 11, 12, 13 ,14]
    fig, axs = plt.subplots(1, len(bins_to_plot), figsize=(18, 8))
    axs = axs.flatten()

    # Adjust spacing to make panels closer together
    fig.subplots_adjust(left=0.08, right=0.98, wspace=0.05)
    for iax, binno in enumerate(bins_to_plot):
        ax = axs[iax]
        ax.plot(mean_qcs[0][binno,:]*1e3, z*1.0e-3, color=casecolors[0], linewidth=2.5, label=casenames[0])
        for idx, m in enumerate(mean_qcs[1:]):
            ax.plot(m[binno,:]*1e3, z*1.0e-3, color=casecolors[idx+1], linewidth=2.5, label=casenames[idx+1])
        ax.set_ylim((0, plot_top))
        ax.set_xlim()
        ax.grid(alpha=0.3)
        if iax != 0:
            ax.set_yticklabels([])
        
        # Add labels only to left column and bottom row
        if iax == 0:
            ax.set_ylabel('Height (km)')
        
        # Simpler title with bin range
        ax.set_title(f'Bin {binno+1}\n'+fr'{10**(bins[binno]):.0f}$\minus${10**(bins[binno+1]):.0f} kg/s', fontsize=14)
        
        # Legend only in first subplot
        if iax == 0:
            ax.legend(loc='upper right', fontsize=11, frameon=True, framealpha=0.9)

    fig.suptitle(r"Mean $q_c$ (10$^{-3}$ g kg$^{-1}$)", fontsize=24)
    plt.tight_layout()
    plt.show()

    bins_to_plot = [10, 11, 12, 13 ,14]
    fig, axs = plt.subplots(1, len(bins_to_plot), figsize=(18, 8))
    axs = axs.flatten()

    # Adjust spacing to make panels closer together
    fig.subplots_adjust(left=0.08, right=0.98, wspace=0.05)
    for iax, binno in enumerate(bins_to_plot):
        ax = axs[iax]
        ax.plot(sum_qcs[0][binno,:]*1e3, z*1.0e-3, color=casecolors[0], linewidth=2.5, label=casenames[0])
        for idx, m in enumerate(sum_qcs[1:]):
            ax.plot(m[binno,:]*1e3, z*1.0e-3, color=casecolors[idx+1], linewidth=2.5, label=casenames[idx+1])
        ax.set_ylim((0, plot_top))
        ax.set_xlim()
        ax.grid(alpha=0.3)
        if iax != 0:
            ax.set_yticklabels([])
        
        # Add labels only to left column and bottom row
        if iax == 0:
            ax.set_ylabel('Height (km)')
        
        # Simpler title with bin range
        ax.set_title(f'Bin {binno+1}\n'+fr'{10**(bins[binno]):.0f}$\minus${10**(bins[binno+1]):.0f} kg/s', fontsize=14)
        
        # Legend only in first subplot
        if iax == 0:
            ax.legend(loc='upper right', fontsize=11, frameon=True, framealpha=0.9)

    fig.suptitle(r"Sum $q_c$ (10$^{-3}$ g kg$^{-1}$)", fontsize=24)
    plt.tight_layout()
    plt.show()

    return

In [ ]:
compare_qc(sum_qcs, mean_qcs, casenames, casecolors, minmf, maxmf, nbins=15)

In [ ]:
compare_wq(sum_wqs, mean_wqs, casenames, casecolors, minmf, maxmf, nbins=15)

In [ ]:
%%script echo skipping

fig = plt.figure(figsize=(6, 9))
ax = fig.add_axes((0.15, 0.1, 0.8, 0.85))
ax.plot((sum_wq_10[ehe18].sum(axis=0) - sum_wq_10[ctl].sum(axis=0))*1.0e3, z*1.0e-3, color='black', linewidth=2.0, label="total diff")
ax.plot(total_wq_diff[0]*1.0e3, z*1.0e-3, linestyle=styles[0], marker=markers[0], label=f'Diff - {pop_labels[0]}', color='black')
ax.plot(total_wq_diff[1]*1.0e3, z*1.0e-3, linestyle=styles[1], marker=markers[1], label=f'Diff - {pop_labels[1]}', color='black')
ax.plot(total_wq_diff[-1]*1.0e3, z*1.0e-3, linestyle=styles[-1], marker=markers[-1], label=f'Diff - {pop_labels[-1]}', color='black')
for bnm in range(10):
    bn =  -1 - bnm
    ax.plot((sum_wq_10[ehe18][bn, :] - sum_wq_10[ctl][bn, :])*1.0e3, z*1.0e-3, color=cc.cm.rainbow(bnm/9), linewidth=1.0, label=f'bin # {len(bins_10)+bn}')
ax.set_ylim((0, 2.5))
ax.set_ylabel('Height (km)')
ax.set_xlim((-15, 2))
ax.set_xticks(np.arange(-15, 2, 3))
ax.set_xlabel(r'($10^{-3}$ g/kg m/s)')
plt.legend(fontsize=14)
plt.show()

In [ ]:
%%script echo skipping

fig = plt.figure(figsize=(6, 9))
ax = fig.add_axes((0.15, 0.1, 0.8, 0.85))
ax.plot((sum_wq_10[ehe18].sum(axis=0) - sum_wq_10[ctl].sum(axis=0))*1.0e3, z*1.0e-3, color='black', linewidth=2.0, label="total diff")
# ax.plot(total_wq_diff[-1]*1.0e3, z*1.0e-3, linestyle=styles[-1], marker=markers[-1], label=f'Diff - {pop_labels[-1]}', color='black')
for bnm in range(10):
    bn =  -1 - bnm
    ax.plot((sum_wq_10[ehe18][bn, :] - sum_wq_10[ctl][bn, :])*1.0e3, z*1.0e-3, color=cc.cm.rainbow(bnm/9), linewidth=2.0, label=f'bin # {len(bins_10)+bn}')
ax.set_ylim((0, 2.5))
ax.set_ylabel('Height (km)')
ax.set_xlim((-10, 2))
ax.set_xticks(np.arange(-10, 2, 2))
ax.set_xlabel(r'($10^{-3}$ g/kg m/s)')
plt.legend(fontsize=14)
plt.show()